# PM2.5 in Northern Thailand — 01 · Fetch and Explore

**DS-270702 Homework 4** · Master of Science in Data Science, Chiang Mai University

โน้ตบุ๊กนี้ทำสองอย่าง: **ดึงข้อมูลจาก API ทุกแหล่ง** และ **ตอบ checkpoint C1–C6** ในใบงาน
ตรรกะจริงอยู่ใน `src/*.py` ทั้งหมด — โน้ตบุ๊กแค่ import แล้วเรียกใช้
เพราะใบงานบังคับ repo layout เป็นสคริปต์ ไม่ใช่โน้ตบุ๊ก (Section 2) แต่ MJ ยังต้องการที่สำรวจข้อมูลแบบเห็นภาพ
วิธีนี้ได้ทั้งสองอย่าง และตอนอัดคลิป 10 นาทีก็เปิดโน้ตบุ๊กนี้ไล่ทีละเซลล์ได้เลย

| Notebook cell | ตรงกับใบงานข้อไหน |
|---|---|
| §1 Configuration | Part A · "record what you fetched" |
| §2 Fetch | Part A · fetch with code, save raw before touching |
| §3 **C4 grid test** | C4 · **รันก่อนอย่างอื่น** เพราะมันตัดสินว่าออกแบบเชิงพื้นที่ได้แค่ไหน |
| §4 C1 Time | C1 · daily averages เป็นวันไทยจริงไหม |
| §5 Prepare + C2 Join | C2 · จำนวนแถวก่อน–หลัง join |
| §6 C3 Missing | C3 · 0.00% missing แปลว่าอะไร |
| §7 C4 full comparison | C4 · สองพื้นที่ให้ข้อมูลต่างกันจริงไหม |
| §8 Figures | Part B · รูปที่ตอบคำถามจริง |
| §9 C6 Ground truth | C6 · CAMS vs Air4Thai |
| §10 What this does not support | Part D · "what are you extrapolating?" |

---

### สามอย่างที่ต้องรู้ก่อนเริ่ม

**1 · แหล่งข้อมูลหลักไม่ใช่ของที่วัดมา**
Open-Meteo Air Quality คือ output ของ Copernicus **CAMS** ซึ่งเป็นแบบจำลองบรรยากาศระดับโลก
มันผลิตค่าให้ทุกกริดทุกชั่วโมงไม่ว่าจะมีเครื่องวัดตรงนั้นหรือไม่ ผลคือ **missing 0.00% พอดี** (ดู §6)
ทุกข้อสรุปเรื่อง "จำนวนวันเกินมาตรฐาน" ในรายงานคือจำนวนวันเกินมาตรฐาน *ของแบบจำลอง*

**2 · ความละเอียดเชิงพื้นที่คือ 0.4° (~45 กม.)**
เชียงใหม่ทั้งจังหวัดกว้างประมาณ 140 กม. → จุดที่ห่างกันไม่ถึง ~45 กม. จะตกกริดเดียวกันและได้ข้อมูล**เหมือนกันทุกตัวเลข**
§3 ทดสอบเรื่องนี้ก่อนเขียนโค้ดวิเคราะห์อะไรทั้งสิ้น ถ้ามันยุบรวมกันจริง นั่นคือ *finding* ที่ต้องเขียนลงรายงาน ไม่ใช่ความล้มเหลว

**3 · ห้ามใช้สภาพอากาศ "ของพรุ่งนี้" จาก ERA5 เป็น feature**
ERA5 เป็น *reanalysis* — มันดูดข้อมูลตรวจวัดที่เกิดขึ้น *หลัง* เวลานั้นเข้ามาด้วย
ใช้ลมพรุ่งนี้จาก ERA5 ทำนาย PM2.5 พรุ่งนี้ = **data leakage** ผิด Rule 2 ตรงๆ และคะแนนที่ได้จะสวยแบบปลอมๆ
โปรเจกต์นี้จึงดึง **Historical Forecast API** เพิ่ม (คือ "โมเดลพยากรณ์ไว้ว่ายังไง" ไม่ใช่ "จริงๆ แล้วเป็นยังไง")
และมี `leakage_guard()` ใน `model.py` ที่จะ **โยน error** ถ้ามี feature ต้องห้ามหลุดเข้ามา

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import config
import checks

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
%matplotlib inline

print("project root :", ROOT)
print("pandas       :", pd.__version__)

---
## 1 · Configuration — what will be fetched

Part A ของใบงานสั่งว่า *"Record what you fetched: source, endpoint, parameters, date range, number of rows, date of retrieval."*
ทุกค่าอยู่ใน `src/config.py` ที่เดียว ไม่มี URL หรือวันที่ hard-code กระจายอยู่ที่อื่น
เซลล์นี้พิมพ์ออกมาเพื่อ copy ลงหัวข้อ Method ในรายงานได้เลย

In [ ]:
print(f"date range : {config.START_DATE}  ..  {config.END_DATE}")
print(f"timezone   : {config.TZ}")
print(f"standard   : {config.PM25_STANDARD} ug/m3 (Thai 24-hour, in force since 1 June 2023)")
print(f"overwrite  : {config.OVERWRITE}   (False = reuse the raw files already in data/raw/)")

print("\nEndpoints")
for k, v in [("air quality (CAMS)", config.AIR_ENDPOINT),
             ("weather archive (ERA5)", config.ARCHIVE_ENDPOINT),
             ("historical FORECAST", config.FORECAST_ARCHIVE_ENDPOINT),
             ("Air4Thai (measured)", config.AIR4THAI_ENDPOINT),
             ("NASA FIRMS", config.FIRMS_ENDPOINT)]:
    print(f"  {k:<24s} {v}")

print("\nLocations")
locs = pd.DataFrame(config.LOCATIONS).T
display(locs)

print(f"\nAir quality variables : {config.AIR_VARS}")
print(f"\nWeather variables ({len(config.WEATHER_VARS.split(','))}):")
for v in config.WEATHER_VARS.split(","):
    star = "  <-- the key addition: basin trapping is invisible without it" if v == "boundary_layer_height" else ""
    print(f"    {v}{star}")

---
## 2 · Fetch

รันสคริปต์จริง ไม่ใช่กดปุ่มดาวน์โหลด (Part A บังคับ) และ **เซฟ raw response ก่อนแตะต้องอะไรทั้งสิ้น**

`OVERWRITE = False` แปลว่ารันซ้ำจะใช้ไฟล์ใน `data/raw/` เดิม → ผลลัพธ์ reproducible จาก clean clone
ยกเว้น **Air4Thai** ที่คืนค่า *ปัจจุบัน* อย่างเดียว ไม่มี history endpoint
→ รันคนละวันได้ไฟล์คนละแบบแน่นอน นี่คือคำตอบของ *"if running it twice produces different files, say why"* ในใบงาน

> ครั้งแรกใช้เวลาสักพัก (หลายสิบ call) ครั้งต่อไปจะเร็วเพราะอ่านจาก cache
> ถ้ายังไม่มี FIRMS map key ให้ข้ามได้ สคริปต์จะเตือนแล้วไปต่อ ไม่ล้ม

In [ ]:
RUN_FETCH = False   # ตั้งเป็น True เพื่อดึงข้อมูลจริง (ครั้งแรกต้องเป็น True)

if RUN_FETCH:
    import fetch_data
    fetch_data.main()
else:
    print("RUN_FETCH = False -- ใช้ไฟล์ที่มีอยู่แล้วใน data/processed/")
    for p in sorted(config.PROCESSED.glob("*.csv")):
        print(f"  {p.name:<36s} {p.stat().st_size/1e6:8.2f} MB")

In [ ]:
# ---- Part A evidence table: exactly what was fetched -----------------------
log_path = config.PROCESSED / "fetch_log.csv"
if log_path.exists():
    log = pd.read_csv(log_path)
    print(f"{len(log)} API calls recorded\n")
    display(log[["source_name", "location", "start_date", "end_date",
                 "rows_returned", "retrieved_at", "status"]].head(20))
    print("\nRows per source:")
    display(log.groupby("source_name")["rows_returned"].agg(["count", "sum"]))
else:
    print("No fetch_log.csv yet -- set RUN_FETCH = True above.")

---
## 3 · C4 first: does this data have the resolution to answer a spatial question?

**รันเซลล์นี้ก่อนเขียนโค้ดวิเคราะห์อะไรก็ตาม**

ใบงานเขียนไว้ว่า *"If your analysis compares two or more locations, demonstrate that they actually return different data before you draw any spatial conclusion."*

CAMS Global ทำงานบนกริด **0.4 องศา ≈ 45 กม.** Open-Meteo จะ **snap** พิกัดที่เราขอไปยังจุดศูนย์กลางกริดที่ใกล้ที่สุด
แล้ว**คืนพิกัดกริดนั้นกลับมาใน response** — นั่นคือหลักฐานตรงๆ

ถ้าเมือง กับ สันทราย (ห่างกัน ~15 กม.) คืน `latitude`/`longitude` เดียวกัน แปลว่าเป็นกริดเดียวกัน
และการเทียบ "เมือง vs ชานเมือง" จากข้อมูลนี้คือการเทียบตัวเลขชุดเดียวกันกับตัวเอง

เซลล์นี้ยิง API จริงวันเดียว จุดละ 1 call — เบามาก แต่ตอบคำถามที่แพงที่สุดในโปรเจกต์

In [ ]:
import requests

test_points = {
    "cm_mueang    (urban)":      (18.7883, 98.9853),
    "cm_san_sai   (~15 km NE)":  (18.9100, 99.0500),
    "cm_hang_dong (~12 km SW)":  (18.6883, 98.9214),
    "cm_mae_chaem (~90 km W)":   (18.5000, 98.3667),
    "cm_chiang_dao(~70 km N)":   (19.3667, 98.9667),
    "cm_omkoi     (~130 km S)":  (17.7947, 98.3644),
}

rows = []
for name, (la, lo) in test_points.items():
    r = requests.get(config.AIR_ENDPOINT, params={
        "latitude": la, "longitude": lo, "hourly": "pm2_5",
        "start_date": "2024-03-15", "end_date": "2024-03-15",
        "timezone": config.TZ}, timeout=60).json()
    rows.append({
        "point": name, "asked_lat": la, "asked_lon": lo,
        "grid_lat": r["latitude"], "grid_lon": r["longitude"],
        "elevation_m": r.get("elevation"),
        "first_4_hours": str(r["hourly"]["pm2_5"][:4]),
    })

grid = pd.DataFrame(rows)
grid["cell"] = list(zip(grid["grid_lat"], grid["grid_lon"]))
display(grid)

n_asked, n_cells = len(grid), grid["cell"].nunique()
print(f"\nพิกัดที่ขอ {n_asked} จุด  ->  ได้กริดจริง {n_cells} เซลล์")

dupes = grid[grid.duplicated("cell", keep=False)].sort_values("cell")
if len(dupes):
    print("\n*** จุดเหล่านี้ตกกริดเดียวกัน ข้อมูลจะเหมือนกันทุกตัวเลข ***")
    for cell, g in dupes.groupby("cell"):
        print(f"  {cell} :")
        for p in g["point"]:
            print(f"      {p}")
    print("\nสรุป: ห้ามใช้คู่เหล่านี้เทียบกันเชิงพื้นที่")
    print("เขียนลงรายงานว่า 'ความละเอียดของแหล่งข้อมูลหยาบกว่าคำถามที่ถาม'")
    print("แล้วเปลี่ยนไปเทียบระดับจังหวัด (ห่างกัน 100-250 กม.) แทน")
else:
    print("\nทุกจุดอยู่คนละกริด -> เทียบเชิงพื้นที่ได้ แต่ยังต้องระบุข้อจำกัดว่า")
    print("นี่คือกริดข้างเคียงของสนามเดียวกันที่ราบเรียบระดับ 45 กม. ไม่ใช่การวัดสองที่")
    print("ที่เป็นอิสระต่อกัน และ CAMS ไม่มีข้อมูลภูมิประเทศ -> มองไม่เห็นแอ่งเชียงใหม่")

### ถ้ากริดยุบรวมกัน ให้ทำอย่างไร

อย่าทิ้งคำถาม "เมือง vs ชนบท" — **เปลี่ยนแหล่งข้อมูล** ไม่ใช่เปลี่ยนคำถาม

| ทางเลือก | ได้อะไร | ต้องทำอะไร |
|---|---|---|
| **CMU CCDC DustBoy** | เซนเซอร์ราคาประหยัดหลายร้อยตัวทั่วภาคเหนือ **รายชั่วโมง ย้อนหลัง 5 ปี** มีสถานีในแม่แจ่ม (N-191) และเชียงดาว (NH-030, Wplus099) | ขอ API key ที่ `open-api.cmuccdc.org` ต้องรออนุมัติ — **สมัครวันนี้เลย** ใช้อีเมล CMU |
| **OpenAQ S3 archive** | ข้อมูล **วัดจริง** จากสถานี PCD, **ไม่ต้องใช้ key** แต่ดูเหมือนจะจบที่ปี 2022 | `openaq-data-archive.s3.amazonaws.com/records/csv.gz/locationid={id}/year=/month=/` |
| **เทียบระดับจังหวัด** | เชียงใหม่ / เชียงราย / ลำปาง / แม่ฮ่องสอน — ห่างกันพอที่จะคนละกริดแน่นอน | ใช้ข้อมูลที่มีอยู่แล้วได้ทันที |

ทางที่ปลอดภัยที่สุดในเวลาที่จำกัด: **ใช้ระดับจังหวัดเป็นการวิเคราะห์หลัก**
แล้วเขียนใน Limitations ว่าคำถามเมือง–ชนบทต้องใช้ DustBoy ซึ่งเป็นสิ่งที่ *"you would need that you do not have"* ตาม Part D พอดี

> **คำเตือนเรื่อง DustBoy:** สถานีในอำเภอห่างไกลออฟไลน์บ่อย (ตอนตรวจสอบ N-191 แม่แจ่ม แสดง "ไม่มีข้อมูล")
> ถ้าใช้ ให้ `plot` ความครบถ้วนของข้อมูลรายเดือน **ก่อน** ตัดสินใจออกแบบ
> สถานีชนบทที่มีข้อมูล 40% ในหน้าเผาจะทำลายการเทียบเมือง–ชนบทแบบเงียบๆ

---
## 4 · C1 · Time

> *"Show that your daily averages correspond to Thailand local days. Describe how you verified this rather than asserting it."*

**วิธีพิสูจน์:** ยิง API เดิม วันเดิม ตัวแปรเดิม **สองครั้ง** ครั้งหนึ่ง `timezone=UTC` อีกครั้ง `timezone=Asia/Bangkok`
ถ้าแกนเวลาเป็น local จริง ค่าเดียวกันจะปรากฏเลื่อนไป **7 ชั่วโมงพอดี**

ถ้าไม่ตรวจแล้วผิด: "ค่าเฉลี่ยรายวัน" ทุกค่าในรายงานจะเป็นวัน UTC ที่ติดป้ายว่าเป็นวันไทย
ยอดฝุ่นช่วงกลางคืนไทยจะถูกหั่นไปอยู่คนละวัน และ **จำนวนวันเกินมาตรฐานจะผิด**

เซลล์นี้ยังเช็คด้วยว่า **สอง endpoint มีแกนเวลาตรงกัน** (ใบงานสั่ง: *"If you fetched from two endpoints, show that their time axes agree"*)

In [ ]:
c1 = checks.c1_time()

---
## 5 · Prepare, and C2 · The join

> *"How many rows did each source have before joining, and how many after? If the number changed, account for every row."*

`prepare_data.py` ทำสองการตัดสินใจที่ต้องปกป้องได้ในรายงาน:

**ก. รายชั่วโมง → รายวัน** เก็บเฉพาะวันที่มีข้อมูล **อย่างน้อย 18 จาก 24 ชั่วโมง**
วันที่ขาดเกินนั้นถูก *ตัดทิ้ง* ไม่ใช่เฉลี่ย เพราะ "ค่าเฉลี่ยรายวัน" จาก 6 ชั่วโมง ไม่ใช่ของชนิดเดียวกับที่มาจาก 24 ชั่วโมง
และจะทำให้จำนวนวันเกินมาตรฐานเพี้ยนแบบเงียบๆ — **นี่คือ "data quality problem ที่เจอและแก้" ตาม Rule 7**

**ข. ทิศลมเฉลี่ยแบบเวกเตอร์** ค่าเฉลี่ยเลขคณิตของ 350° กับ 10° ได้ 180° (ทิศตรงข้าม) ซึ่งผิด
โค้ดแปลงเป็นเวกเตอร์หนึ่งหน่วยก่อนเฉลี่ย

**ค. shift(-1) ข้ามช่องว่าง** ถ้าวันที่หายไปกลางชุดข้อมูล `shift(-1)` จะจับคู่วันที่ไม่ติดกันเข้าด้วยกันอย่างเงียบๆ
โค้ดตรวจว่าแถวถัดไปคือ t+1 จริง ถ้าไม่ใช่ให้ target เป็น NaN

In [ ]:
import prepare_data
prepare_data.main()

In [ ]:
c2 = checks.c2_join()

In [ ]:
# ---- ดูหน้าตาข้อมูลที่ได้ --------------------------------------------------
panel = pd.read_csv(config.PROCESSED / "daily_panel.csv", parse_dates=["date"])
primary = panel[panel["location"] == config.PRIMARY_LOCATION]

print(f"daily_panel : {panel.shape[0]:,} rows x {panel.shape[1]} columns")
print(f"              {panel['location'].nunique()} locations, "
      f"{panel['date'].min().date()} .. {panel['date'].max().date()}")

display(primary[["date", "pm2_5", "pm25_max", "wx_temperature_2m",
                 "wx_relative_humidity_2m", "wx_wind_speed_10m",
                 "wx_precip_total_mm", "pm25_tomorrow", "exceed_tomorrow"]].head(10))

print("\nDescriptive statistics, daily mean PM2.5 by location (ug/m3):")
display(panel.groupby("location")["pm2_5"].describe().round(2))

---
## 6 · C3 · Missing values

> *"Report the percentage missing for every column. If any column is exactly 0.00% missing across years of data, explain what that implies about where the data came from."*

อาจารย์ตั้งคำถามนี้เพราะรู้อยู่แล้วว่าคำตอบคืออะไร นี่คือกับดักที่ตั้งใจวางไว้ — และเป็นจุดที่เชื่อมไปหา C6

`pm2_5`, `pm10`, `carbon_monoxide`, `dust` จะ missing **0.0000% พอดี** ตลอดสามปีกว่า
ไม่มีเครื่องวัดตัวไหนในโลกทำแบบนั้นได้ เครื่องจริงเสียข้อมูลจากการสอบเทียบ ไฟดับ สัญญาณขาด และการซ่อมบำรุง
(Air4Thai เข้ารหัสช่องว่างพวกนั้นเป็น **`-1`** ไม่ใช่ null — ถ้าเฉลี่ยโดยไม่แปลงก่อนจะได้ตัวเลขที่ผิดแบบเงียบๆ)

ข้อมูลที่ครบ 100% คือลายเซ็นของ **model output**

In [ ]:
miss = checks.c3_missing()

In [ ]:
# missing ที่มีจริง ล้วนเกิดจาก pipeline นี้เอง ไม่ใช่จากแหล่งข้อมูล
nz = miss[miss > 0]
print("Columns with missing values, and why each one has them:\n")
for col, pct in nz.items():
    if col.endswith(("_lag1", "_lag2", "_lag3", "_roll3", "_roll7")):
        why = "undefined at the start of each location's series (by construction)"
    elif "tomorrow" in col or col == "is_transition":
        why = "undefined on the last day, and across any gap in the date index"
    elif col.startswith("fc_"):
        why = "historical forecast archive does not cover the full period"
    elif col.startswith("hotspots"):
        why = "no FIRMS detection that day (filled with 0 = no fire detected)"
    else:
        why = "CHECK THIS ONE -- it is not explained by the pipeline"
    print(f"  {col:<42s} {pct:7.3f}%   {why}")

---
## 7 · C4 · Full pairwise comparison

§3 ตรวจกริดจาก API สดๆ เซลล์นี้ตรวจ **ชุดข้อมูลจริงที่ดึงมาแล้ว** ทุกคู่:
ผลต่างสูงสุด, ผลต่างเฉลี่ย และสหสัมพันธ์

คู่ที่มี `max_abs_diff = 0.0` คือชุดข้อมูลเดียวกัน → ตัดออกจากการวิเคราะห์เชิงพื้นที่ และเขียนลงรายงานว่าเจอ

In [ ]:
c4 = checks.c4_places()

---
## 8 · Part B · Figures that answer questions

ใบงานขอ **อย่างน้อย 3 รูป** และเน้นว่า *"A figure nobody discusses is decoration"* และ
*"Every figure needs axis labels with units, a caption, and a sentence in the report saying what a reader should conclude from it."*

`analyse.py` สร้าง 6 รูป แต่ละรูปตอบคำถามที่ใบงานยกตัวอย่างไว้ตรงๆ:

| รูป | ตอบคำถาม |
|---|---|
| fig01 | ฤดูเริ่มและจบเมื่อไหร่ เปลี่ยนไปแต่ละปีไหม |
| fig02 | กี่วันต่อปีที่เกิน 37.5 แนวโน้มขึ้นหรือลง |
| fig03 | ปัญหาเหมือนกันทุกที่ไหม |
| fig04 | สภาพอากาศแบบไหนมากับวันที่แย่ที่สุด |
| fig05 | มี pattern รายสัปดาห์ไหม |
| fig06 | persistence เตือนล่วงหน้าได้แค่ไหน (→ ปูทางไปโมเดล) |

**fig06 คือรูปที่สำคัญที่สุดในโปรเจกต์** มันแสดงว่า persistence เก่งมากโดยรวม แต่พังบนวันที่สถานการณ์เปลี่ยน
ซึ่งเป็นวันเดียวที่ระบบเตือนภัยมีอยู่เพื่อสิ่งนั้น — นั่นคือช่องที่โมเดลจะพิสูจน์ตัวเองได้

In [ ]:
import analyse
analyse.main()

In [ ]:
# ---- แสดงรูปในโน้ตบุ๊ก -----------------------------------------------------
from IPython.display import Image, display as disp

caps = pd.read_csv(config.RESULTS / "figure_captions.csv")
for _, r in caps.iterrows():
    print("=" * 100)
    print(r["figure"])
    print(f"Caption: {r['caption']}")
    disp(Image(filename=str(config.FIGURES / r["file"]), width=980))

In [ ]:
# ---- ตัวเลขที่ต้องยกไปใส่รายงาน ---------------------------------------------
print("Season onset and end, by year (rule: 7-day rolling mean crosses 37.5):")
display(pd.read_csv(config.RESULTS / "season_onset.csv"))

print("\nExceedance days per year:")
display(pd.read_csv(config.RESULTS / "exceedance_by_year.csv"))

print("\nPersistence diagnostics -- this is the argument for building a model at all:")
display(pd.read_csv(config.RESULTS / "persistence_diagnostics.csv", index_col=0))

### อ่านตาราง season onset ให้ระวัง

ถ้าปีไหนออกมาเป็นช่วงสั้นผิดปกติ (เช่น 2 วัน) **อย่ารายงานว่า "ปีนั้นแทบไม่มีฤดูเผา"**
มันแปลว่าค่าเฉลี่ยเคลื่อนที่ 7 วันของปีนั้น *แตะ* เส้น 37.5 แบบเฉียดๆ แล้วตกลงมา
กฎที่ใช้เกณฑ์ตายตัวจะเปราะมากตรงบริเวณเส้นแบ่ง

นี่คือของดีสำหรับรายงาน: ลองนิยามใหม่ (เช่น เกณฑ์ 30 หรือ 35 หรือใช้ percentile ของตัวเอง)
แล้วแสดงว่าวันเริ่มฤดูขยับไปเท่าไหร่ **ความอ่อนไหวของนิยามต่อผลลัพธ์ คือการวิเคราะห์**
ไม่ใช่ข้อบกพร่อง และมันตอบ Part D ข้อ *"what does your analysis actually support?"* ได้ตรงๆ

---
## 9 · C6 · Ground truth

> *"Compare your data against Air4Thai station readings for at least one point in time. Report the difference. If they disagree, which one is right, and what does that mean for your conclusions?"*

**คำตอบว่าอันไหนถูก: Air4Thai** มันคือเครื่องมือวัดที่จุดหนึ่ง ส่วน CAMS คือค่าเฉลี่ยของกริด ~45 กม. จากแบบจำลองระดับโลก

**ข้อจำกัดที่ต้องเขียน:** Air4Thai ไม่มี history endpoint → เทียบได้แค่ ณ เวลาที่รันเท่านั้น
วิธีแก้ที่ทำได้จริง: **รันเซลล์นี้หลายวัน** ผลจะถูก append ต่อท้ายไฟล์เดิม
สะสมสัก 5–10 จุดเวลาแล้วรายงาน bias เฉลี่ย จะหนักแน่นกว่าจุดเดียวมาก

สิ่งที่ต้องรับมือในโค้ด: ค่า `-1` = missing (ไม่ใช่ศูนย์), ทุก field เป็น string รวมพิกัด, และ SSL cert chain ไม่ครบ (`verify=False`)

In [ ]:
c6 = checks.c6_ground_truth()

In [ ]:
# ---- ผลสะสมจากทุกครั้งที่เคยรัน ---------------------------------------------
gt_path = config.RESULTS / "c6_ground_truth.csv"
if gt_path.exists():
    gt = pd.read_csv(gt_path)
    print(f"paired observations accumulated so far: {len(gt)}")
    print(f"distinct snapshot hours: {gt['hour'].nunique()}")
    print(f"\nmean bias (CAMS - measured) : {gt['diff_cams_minus_measured'].mean():+.2f} ug/m3")
    print(f"mean absolute difference    : {gt['diff_cams_minus_measured'].abs().mean():.2f} ug/m3")
    print(f"correlation                 : {gt['pm25_cams'].corr(gt['pm25_measured']):.3f}")

    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    lim = max(gt[["pm25_measured", "pm25_cams"]].max()) * 1.1
    ax.plot([0, lim], [0, lim], color="#b9b8b2", lw=1.4, ls="--", label="1:1 (perfect agreement)")
    ax.scatter(gt["pm25_measured"], gt["pm25_cams"], s=64, color="#2a78d6",
               edgecolor="white", lw=1.5, zorder=3)
    ax.axvline(37.5, color="#eb6834", lw=1, ls=":")
    ax.axhline(37.5, color="#eb6834", lw=1, ls=":")
    ax.set_xlabel(r"Air4Thai measured PM2.5 ($\mu$g/m$^3$)")
    ax.set_ylabel(r"CAMS modelled PM2.5 ($\mu$g/m$^3$)")
    ax.set_title("C6 - modelled against measured")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.legend(frameon=False)
    ax.grid(alpha=.4)
    fig.tight_layout()
    fig.savefig(config.FIGURES / "fig09_ground_truth.png", dpi=200, bbox_inches="tight")
    plt.show()
    print("\nจุดที่ตกคนละฝั่งของเส้นประสีส้ม คือวันที่แบบจำลองกับเครื่องวัด")
    print("'ไม่เห็นตรงกัน' ว่าวันนั้นเกินมาตรฐานหรือไม่ -- นับจำนวนแล้วรายงานไปเลย")
else:
    print("ยังไม่มีไฟล์ -- รันเซลล์ด้านบนก่อน")

---
## 9.5 · Part C · The model

`model.py` ทำสี่อย่าง ตาม Rule 1–7 ใน Section 6 ของใบงาน

1. **ประกาศ target ให้ชัด** — ค่าเฉลี่ย PM2.5 รายวันของวัน t+1 ที่เมืองเชียงใหม่ และจะเกิน 37.5 ไหม
   โดยระบุ **ช่วงเวลาที่ทำนาย = สิ้นวัน t**
2. **`leakage_guard()`** — ถ้ามี feature ที่รู้ไม่ได้ ณ เวลานั้นหลุดเข้ามา จะ **โยน error ทันที** ไม่ใช่ปล่อยให้ได้คะแนนสวยแบบปลอมๆ
3. **ให้คะแนน baseline ก่อน** — persistence (พรุ่งนี้เท่าวันนี้) และ majority class
4. **เลือก threshold ของระบบเตือนภัยอย่างจงใจ** ไม่ปล่อยไว้ที่ 0.5

> **สิ่งที่ต้องดูตอนรัน:** โมเดลจะ**แพ้ baseline บนค่าเฉลี่ยรวม** หรือชนะแค่นิดเดียว
> แต่**ชนะชัดบนวันที่สถานการณ์เปลี่ยน** — ซึ่งเป็นวันเดียวที่ระบบเตือนภัยมีอยู่เพื่อสิ่งนั้น
> ใบงานเขียนไว้ว่า *"If your model does not beat the baseline, say so. That is a finding."*

In [ ]:
import model
model.main()

### อ่านผลยังไง

**Regression** — ดู 3 คอลัมน์ ไม่ใช่คอลัมน์เดียว
`all` คือค่าเฉลี่ยที่ถูกครองโดยวันเงียบๆ · `transition days` คือวันที่ข้ามเส้น 37.5 ซึ่งเป็นวันที่มีค่าจริง

**Classification** — ห้ามดู accuracy วันเกินมาตรฐานมีแค่ ~10% ทายว่า "ปลอดภัย" ทุกวันได้ 0.90 แต่ไร้ประโยชน์
ดู **recall** และดู confusion matrix ว่า **พลาดวันอันตรายกี่วัน**

In [ ]:
from IPython.display import Image, display as disp
for f in ["fig07_regression_vs_baseline.png",
          "fig08_classification_threshold.png",
          "fig10_emission_vs_outcome.png"]:
    path = config.FIGURES / f
    if path.exists():
        print("=" * 100); print(f)
        disp(Image(filename=str(path), width=980))

In [ ]:
# ---- ตัวเลขสุดท้ายที่ต้องยกไปใส่รายงาน --------------------------------------
import json
m = json.load(open(config.RESULTS / "metrics.json", encoding="utf-8"))
r, c = m["regression"], m["classification"]

print(f"train n={m['n_train']}   test n={m['n_test']}   features={m['n_features']}\n")
print(f"{'REGRESSION':<24s} {'all':>8s} {'season':>8s} {'transition':>11s}")
for k in ["baseline_persistence", "ridge", "hgb"]:
    v = r[k]
    print(f"  {k:<22s} {v['mae']:8.3f} {v.get('mae_burning_season', float('nan')):8.3f} "
          f"{v.get('mae_transition_days', float('nan')):11.3f}")

op = c.get("operating_point_recall90")
if op:
    cm = op["confusion_matrix"]
    print(f"\nWARNING SYSTEM at threshold {op['threshold']:.3f}")
    print(f"  caught {cm[1][1]} of {cm[1][1] + cm[1][0]} exceedance days")
    print(f"  MISSED {op['missed_dangerous_days']}   false alarms {op['false_alarms']}")
    print(f"  recall {op['recall']:.3f}   precision {op['precision']:.3f}")

cb = c["baseline_persistence"]
print(f"\n  persistence baseline for comparison: recall {cb['recall']:.3f}, "
      f"precision {cb['precision']:.3f}")

---
## 10 · What this data does and does not support

ส่วนนี้ตอบ Part D โดยตรง: *"What does your analysis actually support, and what are you extrapolating?"*
และเป็นจุดที่ใบงานให้คะแนนสูงสุด (Recommendation 20 คะแนน)

### สามระดับที่ตั้งใจไว้ตอนแรก — ประเมินตามความเป็นจริงของข้อมูล

| ระดับ | ทำได้ไหม | เหตุผล |
|---|---|---|
| **1 · Seasonal predict** | **ไม่ได้** ในฐานะโมเดลพยากรณ์ | มีข้อมูลตั้งแต่ 2023-01-01 → มีฤดูเผาแค่ **4 ฤดู** จะ train หรือ validate อะไรก็ได้ n=4 ทั้งนั้น ใส่ ENSO/IOD เข้าไปก็ยัง validate ไม่ได้ **สิ่งที่ทำได้และได้คะแนน: climatology + season onset detection** (fig01) แล้วเขียนตรงๆ ว่าข้อมูลสนับสนุนได้แค่การบรรยาย ไม่ใช่การพยากรณ์ฤดูกาล |
| **2 · Short-term predict** | **ได้** และเป็นแกนของงาน | t+1 daily mean, baseline = persistence, feature ทุกตัวรู้ได้ ณ เวลาทำนาย |
| **3 · Caution predict** | **ได้** และตรงกับ classification ในใบงาน | เกิน 37.5 พรุ่งนี้ไหม, imbalanced ~9-10% → รายงาน recall เป็นหลัก ไม่ใช่ accuracy |

### สิ่งที่ข้อมูลชุดนี้ **ไม่** สนับสนุน — เขียนลงหัวข้อ Limitations

1. **จำนวนวันเกินมาตรฐานที่แท้จริง** — ทุกตัวเลขคือของ CAMS ขนาดของ bias อยู่ใน C6
2. **แนวโน้มระยะยาว** — 4 ปีไม่ใช่ trend และปี 2026 ยังไม่จบ อย่าเทียบปีเต็มกับปีที่ยังไม่ครบ
3. **ความแตกต่างเมือง–ชนบทในเชียงใหม่** — ถ้า §3 พบว่ากริดยุบรวม (ดู §3 สำหรับทางแก้)
4. **สาเหตุ** — สหสัมพันธ์ระหว่าง hotspot กับ PM2.5 ไม่ได้พิสูจน์ว่าไฟจุดไหนทำให้ฝุ่นที่ไหน ต้องใช้แบบจำลองการเคลื่อนที่ของอากาศ
5. **ผลของมาตรการรัฐ** — ข้อมูลนี้บอกไม่ได้ว่ามาตรการห้ามเผาได้ผลหรือไม่ ต้องมีข้อมูลการบังคับใช้และกลุ่มเปรียบเทียบ

### สิ่งที่จะทำให้ข้อเสนอแนะแข็งขึ้น (Part D ถามตรงๆ ว่า *"what would you need that you do not have?"*)

- **ข้อมูลวัดจริงย้อนหลังหลายปีระดับอำเภอ** → CMU CCDC DustBoy (ต้องขอ key)
- **ข้อมูลผู้ป่วยทางเดินหายใจรายวันของเชียงใหม่** → HDC ต้องมีอาจารย์รับรอง; **NHSO (สปสช.) เปิดสาธารณะ** เป็นทางที่ทำได้จริง
- **สถิตินักท่องเที่ยวรายเดือนรายจังหวัด** → กระทรวงการท่องเที่ยวและกีฬา ไฟล์ .xlsx รายปี
- **บันทึกการปิดโรงเรียน** → ไม่พบว่ามีการรวบรวมไว้เป็นชุดข้อมูล นั่นเองก็เป็น finding

รายละเอียดทั้งหมด พร้อม URL และวันที่ของแหล่งอ้างอิง อยู่ใน `docs/`:
`RESEARCH_prevention.md`, `RESEARCH_recovery.md`, `RESEARCH_data_sources.md`

---

## ขั้นต่อไป

```bash
python src/model.py     # baseline, models, C5, และ threshold ของระบบเตือนภัย
```

แล้วอ่าน `docs/REPORT_GUIDE.md` ซึ่งแมป checkpoint และเกณฑ์ให้คะแนนทุกข้อ
ไปยังไฟล์ที่มีตัวเลขคำตอบอยู่